# Notebook 3: Gene Expression Cancer Classification

**BMI 503/511 — Machine Learning: Classification Methods**  
**Instructor:** Pratik Dutta, Ph.D. | Stony Brook University

---

## Learning Objectives
1. Work with **high-dimensional genomic data** (many features, few samples)
2. Apply **PCA** for dimensionality reduction and visualization
3. Build a complete **sklearn Pipeline** (scaling → PCA → classifier)
4. Perform **feature selection** to find cancer-relevant genes
5. Understand the **p >> n** challenge in genomics

**Estimated time: ~30 minutes**

## 1. Load Gene Expression Data

We use two datasets from sklearn that simulate common genomics classification problems:

1. **Simulated RNA-seq-like data** (make_classification) — to understand the pipeline
2. **Real gene expression structure** — using sklearn's dataset generators with genomics-realistic dimensions

In real genomics:
- Thousands of genes (features) but only tens-to-hundreds of samples
- This is the **p >> n problem** (more features than samples)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# Simulate gene expression data
# 200 samples, 5000 genes, but only ~50 are truly informative
np.random.seed(42)

X, y = make_classification(
    n_samples=200,
    n_features=5000,       # 5000 "genes"
    n_informative=50,      # Only 50 genes matter
    n_redundant=100,       # 100 are correlated copies
    n_classes=2,           # Tumor vs Normal
    class_sep=1.5,
    random_state=42
)

# Create gene names
gene_names = [f'Gene_{i:04d}' for i in range(5000)]
df = pd.DataFrame(X, columns=gene_names)
df['Label'] = y
df['Class'] = df['Label'].map({0: 'Normal', 1: 'Tumor'})

print(f"Dataset dimensions: {X.shape[0]} samples × {X.shape[1]} genes")
print(f"Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}")
print(f"\n⚠️ This is a classic p >> n problem:")
print(f"   {X.shape[1]} features but only {X.shape[0]} samples")
print(f"   Ratio: {X.shape[1] / X.shape[0]:.0f}x more features than samples!")

## 2. PCA for Visualization

We can't visualize 5000 dimensions, but PCA can compress them into 2D.

**PCA** finds the directions of maximum variance in the data.

In [ ]:
# Standardize first
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA to 2 components for visualization
pca_viz = PCA(n_components=2)
X_pca2 = pca_viz.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: PCA scatter
colors_map = {0: '#3B82F6', 1: '#EF4444'}
for label, name in [(0, 'Normal'), (1, 'Tumor')]:
    mask = y == label
    axes[0].scatter(X_pca2[mask, 0], X_pca2[mask, 1],
                    c=colors_map[label], label=name, alpha=0.7, s=40)
axes[0].set_xlabel(f'PC1 ({pca_viz.explained_variance_ratio_[0]:.1%} variance)', fontsize=11)
axes[0].set_ylabel(f'PC2 ({pca_viz.explained_variance_ratio_[1]:.1%} variance)', fontsize=11)
axes[0].set_title('PCA — 5000 Genes → 2D', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(alpha=0.3)

# Right: Scree plot (variance explained)
pca_full = PCA(n_components=50)
pca_full.fit(X_scaled)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)

axes[1].plot(range(1, 51), cumvar, 'o-', color='#0D9488', markersize=4)
axes[1].axhline(0.95, color='#EF4444', linestyle='--', alpha=0.7, label='95% variance')
n_95 = np.argmax(cumvar >= 0.95) + 1
axes[1].axvline(n_95, color='#F59E0B', linestyle='--', alpha=0.7, label=f'{n_95} PCs for 95%')
axes[1].set_xlabel('Number of Principal Components', fontsize=11)
axes[1].set_ylabel('Cumulative Variance Explained', fontsize=11)
axes[1].set_title('Scree Plot — How Many PCs Do We Need?', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 The first 2 PCs capture {pca_viz.explained_variance_ratio_.sum():.1%} of variance.")
print(f"   We need {n_95} PCs to capture 95% — a 5000→{n_95} compression!")

## 3. Building a Classification Pipeline

In sklearn, a **Pipeline** chains preprocessing and modeling steps.  
This ensures:
- No data leakage (scaling is fit only on training data)
- Clean, reproducible code
- Easy to swap components

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Define pipelines
pipelines = {
    'LogReg (no PCA)': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=2000, C=0.1, random_state=42))
    ]),
    'LogReg + PCA(50)': Pipeline([
        ('scaler', StandardScaler()),
        ('pca', PCA(n_components=50)),
        ('clf', LogisticRegression(max_iter=2000, random_state=42))
    ]),
    'RF (no PCA)': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
    ]),
    'RF + PCA(50)': Pipeline([
        ('scaler', StandardScaler()),
        ('pca', PCA(n_components=50)),
        ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
    ]),
    'SVM + PCA(50)': Pipeline([
        ('scaler', StandardScaler()),
        ('pca', PCA(n_components=50)),
        ('clf', SVC(kernel='rbf', probability=True, random_state=42))
    ]),
}

# Cross-validation evaluation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"{'Pipeline':<25} {'CV Accuracy':>12} {'CV AUC-ROC':>12}")
print('=' * 52)

cv_results = {}
for name, pipe in pipelines.items():
    acc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='accuracy').mean()
    auc_score = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc').mean()
    cv_results[name] = {'accuracy': acc, 'auc': auc_score}
    print(f"{name:<25} {acc:>12.4f} {auc_score:>12.4f}")

print("\n💡 Key insight: PCA often HELPS with high-dimensional data by reducing noise.")
print("   Without PCA, models may overfit to the 4950 irrelevant genes.")

## 4. Feature Selection — Finding Important Genes

Instead of PCA (which creates new composite features), we can directly select the most informative genes.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif

# Method 1: ANOVA F-test (fast, parametric)
selector_f = SelectKBest(f_classif, k=50)
selector_f.fit(X_train, y_train)

# Method 2: Mutual Information (non-parametric, captures non-linear)
selector_mi = SelectKBest(mutual_info_classif, k=50)
selector_mi.fit(X_train, y_train)

# Get selected gene indices
top_f_genes = np.array(gene_names)[selector_f.get_support()]
top_mi_genes = np.array(gene_names)[selector_mi.get_support()]
overlap = set(top_f_genes) & set(top_mi_genes)

print(f"ANOVA F-test: selected {len(top_f_genes)} genes")
print(f"Mutual Information: selected {len(top_mi_genes)} genes")
print(f"Overlap: {len(overlap)} genes selected by BOTH methods")

# Volcano-style plot: F-score vs gene index
f_scores = selector_f.scores_
fig, ax = plt.subplots(figsize=(12, 4))
ax.scatter(range(len(f_scores)), f_scores, s=3, alpha=0.4, color='#64748B')
top_idx = selector_f.get_support()
ax.scatter(np.where(top_idx)[0], f_scores[top_idx], s=10, alpha=0.8, color='#EF4444', label='Top 50 genes')
ax.set_xlabel('Gene Index', fontsize=11)
ax.set_ylabel('ANOVA F-Score', fontsize=11)
ax.set_title('Gene Importance — ANOVA F-Scores', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print("\n💡 In real genomics, these 'important genes' become biomarker candidates.")
print("   You would then validate them with independent experiments.")

## 5. Pipeline with Feature Selection

In [ ]:
# Compare: PCA vs Feature Selection vs Raw
comparison_pipelines = {
    'RF — All 5000 genes': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(n_estimators=200, random_state=42))
    ]),
    'RF — PCA (50 components)': Pipeline([
        ('scaler', StandardScaler()),
        ('pca', PCA(n_components=50)),
        ('clf', RandomForestClassifier(n_estimators=200, random_state=42))
    ]),
    'RF — Top 50 genes (ANOVA)': Pipeline([
        ('scaler', StandardScaler()),
        ('selector', SelectKBest(f_classif, k=50)),
        ('clf', RandomForestClassifier(n_estimators=200, random_state=42))
    ]),
    'RF — Top 100 genes (ANOVA)': Pipeline([
        ('scaler', StandardScaler()),
        ('selector', SelectKBest(f_classif, k=100)),
        ('clf', RandomForestClassifier(n_estimators=200, random_state=42))
    ]),
}

print(f"{'Strategy':<30} {'CV Accuracy':>12} {'CV AUC':>10}")
print('=' * 55)

strategy_results = []
for name, pipe in comparison_pipelines.items():
    acc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='accuracy').mean()
    auc_s = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc').mean()
    strategy_results.append({'Strategy': name, 'Accuracy': acc, 'AUC': auc_s})
    print(f"{name:<30} {acc:>12.4f} {auc_s:>10.4f}")

# Visual comparison
sr_df = pd.DataFrame(strategy_results)
fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(sr_df))
bars = ax.bar(x, sr_df['AUC'], color=['#64748B', '#8B5CF6', '#EF4444', '#10B981'],
              alpha=0.85, edgecolor='white', linewidth=1.5)
ax.set_xticks(x)
ax.set_xticklabels(sr_df['Strategy'], rotation=15, ha='right', fontsize=10)
ax.set_ylabel('CV AUC-ROC', fontsize=12)
ax.set_title('Dimensionality Reduction Strategy Comparison', fontsize=14, fontweight='bold')
ax.set_ylim([0.8, 1.02])
ax.grid(axis='y', alpha=0.3)

for bar, val in zip(bars, sr_df['AUC']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Final Evaluation on Test Set

In [ ]:
# Train best pipeline on full training set and evaluate on held-out test
best_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('selector', SelectKBest(f_classif, k=50)),
    ('clf', RandomForestClassifier(n_estimators=200, random_state=42))
])

best_pipe.fit(X_train, y_train)
y_pred = best_pipe.predict(X_test)
y_prob = best_pipe.predict_proba(X_test)[:, 1]

print("Final Test Set Evaluation")
print("=" * 45)
print(classification_report(y_test, y_pred, target_names=['Normal', 'Tumor']))
print(f"AUC-ROC: {roc_auc_score(y_test, y_prob):.4f}")

# Confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Normal', 'Tumor'], yticklabels=['Normal', 'Tumor'])
ax.set_title('Final Model — Confusion Matrix', fontsize=13, fontweight='bold')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

## 7. Extracting the Gene Signature

In [ ]:
# Which genes were selected by ANOVA, and what are their RF importances?
selected_mask = best_pipe.named_steps['selector'].get_support()
selected_genes = np.array(gene_names)[selected_mask]
rf_importances = best_pipe.named_steps['clf'].feature_importances_

gene_imp = pd.DataFrame({
    'Gene': selected_genes,
    'RF_Importance': rf_importances
}).sort_values('RF_Importance', ascending=False)

# Top 15 genes
top15 = gene_imp.head(15)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(range(len(top15)), top15['RF_Importance'].values,
               color='#0D9488', alpha=0.85)
ax.set_yticks(range(len(top15)))
ax.set_yticklabels(top15['Gene'].values)
ax.invert_yaxis()
ax.set_xlabel('Random Forest Feature Importance', fontsize=12)
ax.set_title('Top 15 Genes in Cancer Signature', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n🧬 Gene Signature: {len(selected_genes)} genes selected from 5000")
print(f"   Top 5: {', '.join(top15['Gene'].head(5).values)}")
print(f"\n💡 In real research, these genes would be validated with:")
print(f"   - Independent cohort testing")
print(f"   - RT-qPCR or other experimental assays")
print(f"   - Pathway enrichment analysis (GSEA, GO)")
print(f"   - Literature search for known cancer associations")

## ✅ Exercises for Students

1. **Vary the number of selected genes** (k=10, 50, 100, 500). Plot AUC vs k.
2. **Try UMAP** instead of PCA for visualization: `!pip install umap-learn` → `from umap import UMAP`
3. **Multi-class extension**: Change `n_classes=4` in the data generator (simulate multiple cancer subtypes). Which classifier handles multi-class best?
4. **Real data challenge**: Load the `load_digits()` dataset from sklearn (images of handwritten digits). Apply the same pipeline — how does it perform?
5. **Discuss**: Why might feature selection (ANOVA) be preferred over PCA in clinical genomics, even if PCA gives similar accuracy?